In [1]:
import skrub
import pandas as pd
import joblib
from skrub import TableVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

def train_model(df, labels):
    #TODO generate additional features
    vectorizer = TableVectorizer()
    enc = vectorizer.fit_transform(df)
    model = HistGradientBoostingClassifier()
    model.fit(enc, labels)
    joblib.dump(vectorizer, "v.pkl")
    joblib.dump(model, "m.pkl")

def serve(data, path):
    encoder = joblib.load("v.pkl")
    predictor = joblib.load("m.pkl")
    return predictor.predict(encoder.transform(data))

/home/schelter-ldap/miniconda3/envs/py312/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/schelter-ldap/miniconda3/envs/py312/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [7]:
import os
import re
from dotenv import load_dotenv
from litellm import completion

load_dotenv()


def extract_python_code(text: str) -> str:
    match = re.search(r"```python\s*\n(.*?)```", text, re.DOTALL)
    if not match:
        return ""
    return match.group(1).strip()
    
schema = """
    ser (string), 
    status (string), 
    score (float), 
    quantity (int), 
    priority (string), 
    event_date (string), 
    extra (float), 
    flag (bool),
    misc (string),
    label (bool)
"""

code_template = """
import skrub
import pandas as pd
import joblib
from skrub import TableVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

def train_model(df, labels):
    #TODO generate additional features
    vectorizer = TableVectorizer()
    enc = vectorizer.fit_transform(df)
    model = HistGradientBoostingClassifier()
    model.fit(enc, labels)
    joblib.dump(vectorizer, "v.pkl")
    joblib.dump(model, "m.pkl")

def serve(data, path):
    encoder = joblib.load("v.pkl")
    predictor = joblib.load("m.pkl")
    return predictor.predict(encoder.transform(data))
"""

model = "gemini/gemini-2.5-pro"
model_name = "gemini25pro"

for repetition in range(10):
    print(f'Iteration {repetition}...')
    response = completion(
        model=model,
        messages=[{"role": "user", "content": f"""
            Complete the following ML pipeline code with suitable operations from common libraries pandas, torch, etc.  
            The input data has the following schema: {schema}
            INSERT CODE for the #TODO
            RESPOND WITH THE FULL CODE.  
            -----
            
            {code_template}
        """}],
        temperature=0.5,
    )
    
    code = extract_python_code(response.choices[0].message.content)
    #print(code)
    with open(f'experiments/globalprogramsynthesis/{model_name}_traintest_{repetition}.py', "w") as f:
        f.write(code)

Iteration 0...
Iteration 1...
Iteration 2...
Iteration 3...
Iteration 4...
Iteration 5...
Iteration 6...
Iteration 7...
Iteration 8...
Iteration 9...
